# BitcoinPetl API Demo

This notebook demonstrates how to use the functions in `bitcoin_petl_utils.py`
to fetch live Bitcoin prices and perform simple ETL operations with Petl.

## Setup and Imports

In [2]:
!pip petl pandas
import petl as etl
import pandas as pd

from bitcoin_petl_utils import (
    fetch_btc_price_table,
    filter_recent,
    expand_demo_rows,
)

## Fetch & Inspect Raw Table

In [9]:
# Fetch one live row as a Petl table
tbl = fetch_btc_price_table()
print("Raw PETL table from fetch_btc_price_table():")
print(etl.look(tbl))

Raw PETL table from fetch_btc_price_table():
+------------+-----------+
| timestamp  | price_usd |
+============+===========+
| 1747508876 |    103094 |
+------------+-----------+



## Filter Recent Rows

In [10]:
# Demonstrate filter_recent on a single-row table
recent_tbl = filter_recent(tbl, lookback_min=10)
print("\nAfter filter_recent(tbl, 10):")
print(etl.look(recent_tbl))


After filter_recent(tbl, 10):
+------------+-----------+
| timestamp  | price_usd |
+============+===========+
| 1747508876 |    103094 |
+------------+-----------+



## Expand Demo Rows

In [11]:
# Create a 5-row demo table for demonstration purposes
demo_tbl = expand_demo_rows(tbl, n=5, dt=60)
print("\nDemo table with 5 synthetic rows:")
print(etl.look(demo_tbl))


Demo table with 5 synthetic rows:
+------------+-----------+
| timestamp  | price_usd |
+============+===========+
| 1747508636 |    103094 |
+------------+-----------+
| 1747508696 |    103094 |
+------------+-----------+
| 1747508756 |    103094 |
+------------+-----------+
| 1747508816 |    103094 |
+------------+-----------+
| 1747508876 |    103094 |
+------------+-----------+



## Convert to pandas DataFrame

In [12]:
# Convert the PETL table to a pandas DataFrame
df = etl.todataframe(demo_tbl)
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
df.set_index('timestamp', inplace=True)
df.head()

,price_usd
timestamp,
2025-05-17 19:03:56,103094
2025-05-17 19:04:56,103094
2025-05-17 19:05:56,103094
2025-05-17 19:06:56,103094
2025-05-17 19:07:56,103094


## Error Handling Example

In [13]:
# Show error handling by pointing to a bad URL
import importlib
import bitcoin_petl_utils as utils
# Temporarily break the URL
utils.CG_URL = "https://api.coingecko.invalid/foo"
try:
    _ = fetch_btc_price_table()
except Exception as e:
    print("Caught an error as expected:", e)
# Restore original module state
importlib.reload(utils)

Caught an error as expected: HTTPSConnectionPool(host='api.coingecko.invalid', port=443): Max retries exceeded with url: /foo (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7f93ed8a9900>: Failed to resolve 'api.coingecko.invalid' ([Errno -2] Name or service not known)"))


<module 'bitcoin_petl_utils' from '/work/DATA605/Spring2025/projects/TutorTask354_Spring2025_Real_Time_Bitcoin_Price_Analysis_with_Petl/bitcoin_petl_utils.py'>

## Wrap-Up & Takeaways

1. **`fetch_btc_price_table()`**  
   - Fetches a single, real-time Bitcoin price from CoinGecko.  
   - Returns a one-row Petl table with UNIX `timestamp` and `price_usd`.

2. **`expand_demo_rows(tbl, n, dt)`**  
   - Clones that one row into `n` rows, each shifted by `dt` seconds.  
   - Useful for showing ETL operations on multi-row data in tutorials.  
   - In this demo, we generated 5 rows spaced 1 minute apart.

3. **`filter_recent(table, lookback_min)`**  
   - Converts the `timestamp` column to integers and keeps only rows  
     within the last `lookback_min` minutes.  
   - When run on our 5-row demo, it lets you see how filtering works  
     across multiple records rather than just one.

4. **Converting Petl → pandas**  
   - `etl.todataframe()` turns a Petl table into a pandas DataFrame.  
   - After parsing the timestamps into datetime objects, you can  
     leverage pandas’ powerful time-series tools (rolling windows, plotting, etc.).

---

By walking through:

- **a real fetch**,  
- **synthetic multi-row generation**,  
- **time-based filtering**,  
- and **conversion to pandas**,  

you now have a clear recipe for integrating live BTC data into any ETL or analytics pipeline. 